# 元大 ETF 投資組合下載器
自動抓取持股明細並輸出格式化 xlsx，目錄結構：
```
ETF_Investment_Portfolio/
├── 00981A/
│   ├── ETF_Investment_Portfolio_20260602.xlsx
│   └── ...
└── 00403A/
    └── ...
```
台灣時間 晚上6~7 點執行

In [1]:
# ========================================
# 設定：修改此處
# ========================================

ETF_LIST = [
    "00981A",
    "00403A",
    "0050",
    "0056",  # 如需其他 ETF 可在此加入
    "00878",
    "00713",
    "00929",
    "00919",
    "0052",
    
    "009819",
    "009805"
]

BASE_DIR = "ETF_Investment_Portfolio"  # 輸出根目錄

In [2]:
import requests
import pandas as pd
import time
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment

# 日期
today     = datetime.today()
date_str  = today.strftime("%Y%m%d")
roc_year  = today.year - 1911
roc_date  = f"{roc_year}/{today.month:02d}/{today.day:02d}"

print(f"今日日期：{roc_date}（民國）/ {today.strftime('%Y-%m-%d')}")

今日日期：115/07/16（民國）/ 2026-07-16


In [3]:
# ========================================
# 抓取持股資料
# ========================================

URL = "https://www.pocket.tw/api/cm/MobileService/ashx/GetDtnoData.ashx"

def fetch_etf(etf: str) -> pd.DataFrame | None:
    """從 pocket.tw 抓取 ETF 持股明細，回傳 DataFrame。"""
    print(f"\n開始抓取 {etf}")
    params = {
        "action":    "getdtnodata",
        "DtNo":      "59449513",
        "ParamStr":  f"AssignID={etf};MTPeriod=0;DTMode=0;DTRange=1;DTOrder=1;MajorTable=M722;",
        "FilterNo":  "0"
    }
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer":    f"https://www.pocket.tw/etf/tw/{etf}/fundholding"
    }
    try:
        r = requests.get(URL, params=params, headers=headers, timeout=30)
        r.raise_for_status()
        data = r.json()
        if "Title" not in data or "Data" not in data or len(data["Data"]) == 0:
            print(f"  {etf} 無資料")
            return None
        df = pd.DataFrame(data["Data"], columns=data["Title"])
        df["ETF"] = etf
        print(f"  完成 {etf} 共 {len(df)} 筆 | 欄位: {df.columns.tolist()}")
        return df
    except Exception as e:
        print(f"  {etf} 抓取失敗: {e}")
        return None

In [4]:
# ========================================
# 產生 xlsx（與範本格式完全相同）
# ========================================

def find_col(df: pd.DataFrame, candidates: list) -> str | None:
    """依候選清單找欄位名稱（容錯不同 ETF 欄位命名）。"""
    for c in candidates:
        if c in df.columns:
            return c
    return None


def write_xlsx(df: pd.DataFrame, path: Path):
    """將持股 DataFrame 寫入格式化 xlsx。"""
    wb = Workbook()
    ws = wb.active
    ws.title = "基金投資組合"

    # 欄寬
    ws.column_dimensions["A"].width = 24.75
    ws.column_dimensions["B"].width = 24.75
    ws.column_dimensions["C"].width = 12.9
    ws.column_dimensions["D"].width = 10.35

    bold12   = Font(name="Arial", bold=True,  size=12)
    normal12 = Font(name="Arial", bold=False, size=12)
    center   = Alignment(horizontal="center")
    left_a   = Alignment(horizontal="left")
    right_a  = Alignment(horizontal="right")

    def sc(coord, value, font=None, align=None):
        c = ws[coord]
        c.value = value
        if font:  c.font  = font
        if align: c.alignment = align

    # ── 資料日期
    sc("A1", f"資料日期：{roc_date}", font=bold12, align=left_a)

    # ── 基金資產（標題，合併列）
    sc("A3", "基金資產", font=bold12, align=center)
    ws.merge_cells("A3:C3")

    # ── 淨資產 / 流通單位 / 每單位淨值（空白佔位）
    for lc, lv, mr in [
        ("A4", "淨資產",       "B4:C4"),
        ("A5", "流通在外單位數", "B5:C5"),
        ("A6", "每單位淨值",    "B6:C6"),
    ]:
        sc(lc, lv, font=normal12, align=left_a)
        ws.merge_cells(mr)
        sc(mr.split(":")[0], "", font=normal12, align=right_a)

    # ── 基金資產表（期貨 / 股票）
    for coord, val in [("A8","項目"),("B8","金額"),("C8","權重")]:
        sc(coord, val, font=bold12, align=center)
    for coord, val, al in [
        ("A9",  "期貨(名目本金)", left_a),  ("B9",  "", right_a), ("C9",  "", right_a),
        ("A10", "股票",          left_a),  ("B10", "", right_a), ("C10", "", right_a),
    ]:
        sc(coord, val, font=normal12, align=al)

    # ── 現金部位表
    for coord, val in [("A12","項目"),("B12","金額"),("C12","權重")]:
        sc(coord, val, font=bold12, align=center)
    for i, label in enumerate(["現金","期貨保證金","申贖應付款","應收付證券款"]):
        r = 13 + i
        sc(f"A{r}", label, font=normal12, align=left_a)
        sc(f"B{r}", "",    font=normal12, align=right_a)
        sc(f"C{r}", "",    font=normal12, align=right_a)

    # ── 股票區（大標 + 表頭）
    sc("A19", "股票", font=bold12, align=center)
    ws.merge_cells("A19:D19")
    for coord, val in [("A20","股票代號"),("B20","股票名稱"),("C20","股數"),("D20","持股權重")]:
        sc(coord, val, font=bold12, align=center)

    # ── 持股明細（容錯欄位名）
    col_id   = find_col(df, ["標的代號","股票代號","代號"])
    col_name = find_col(df, ["標的名稱","股票名稱","名稱"])
    col_qty  = find_col(df, ["持有數","股數","數量"])
    col_wt   = find_col(df, ["權重(%)","持股權重","權重"])

    for excel_row, (_, row) in enumerate(df.iterrows(), start=21):
        val_id   = row[col_id]   if col_id   else ""
        val_name = row[col_name] if col_name else ""
        val_qty  = row[col_qty]  if col_qty  else ""
        val_wt   = row[col_wt]   if col_wt   else ""

        # 千分位格式
        try:
            val_qty = f"{int(str(val_qty).replace(',',''))  :,}"
        except: pass

        # 百分比格式（兩位小數）
        try:
            val_wt = f"{float(str(val_wt).replace('%','')):.2f}%"
        except: val_wt = str(val_wt)

        sc(f"A{excel_row}", str(val_id),   font=normal12, align=left_a)
        sc(f"B{excel_row}", str(val_name), font=normal12, align=left_a)
        sc(f"C{excel_row}", val_qty,        font=normal12, align=right_a)
        sc(f"D{excel_row}", val_wt,         font=normal12, align=right_a)

    path.parent.mkdir(parents=True, exist_ok=True)
    wb.save(str(path))
    print(f"  已儲存: {path}")

In [5]:
# ========================================
# 主流程：抓取 → 存 CSV → 存 xlsx
# ========================================

base_dir = Path(BASE_DIR)
all_df   = []

for etf in ETF_LIST:

    df = fetch_etf(etf)
    if df is None:
        continue

    # 存格式化 xlsx
    xlsx_path = base_dir / etf / f"ETF_Investment_Portfolio_{date_str}.xlsx"
    write_xlsx(df, xlsx_path)

    all_df.append(df)
    time.sleep(1)

# 合併所有 ETF 為一個 CSV
if all_df:
    final_df = pd.concat(all_df, ignore_index=True)
    final_csv = base_dir / f"ETF_HOLDING_ALL_{date_str}.csv"
    final_df.to_csv(final_csv, index=False, encoding="utf-8-sig")
    print(f"\n====================")
    print(f"全部完成，總筆數：{len(final_df)}")
    print(f"合併 CSV：{final_csv}")
else:
    print("沒有成功取得任何資料")



開始抓取 00981A
  完成 00981A 共 53 筆 | 欄位: ['日期', '標的代號', '標的名稱', '權重(%)', '持有數', '單位', 'ETF']
  已儲存: ETF_Investment_Portfolio\00981A\ETF_Investment_Portfolio_20260716.xlsx

開始抓取 00403A
  完成 00403A 共 53 筆 | 欄位: ['日期', '標的代號', '標的名稱', '權重(%)', '持有數', '單位', 'ETF']
  已儲存: ETF_Investment_Portfolio\00403A\ETF_Investment_Portfolio_20260716.xlsx

開始抓取 0050
  完成 0050 共 52 筆 | 欄位: ['日期', '標的代號', '標的名稱', '權重(%)', '持有數', '單位', 'ETF']
  已儲存: ETF_Investment_Portfolio\0050\ETF_Investment_Portfolio_20260716.xlsx

開始抓取 0056
  完成 0056 共 51 筆 | 欄位: ['日期', '標的代號', '標的名稱', '權重(%)', '持有數', '單位', 'ETF']
  已儲存: ETF_Investment_Portfolio\0056\ETF_Investment_Portfolio_20260716.xlsx

開始抓取 00878
  完成 00878 共 34 筆 | 欄位: ['日期', '標的代號', '標的名稱', '權重(%)', '持有數', '單位', 'ETF']
  已儲存: ETF_Investment_Portfolio\00878\ETF_Investment_Portfolio_20260716.xlsx

開始抓取 00713
  完成 00713 共 51 筆 | 欄位: ['日期', '標的代號', '標的名稱', '權重(%)', '持有數', '單位', 'ETF']
  已儲存: ETF_Investment_Portfolio\00713\ETF_Investment_Portfolio_20260716.xlsx

開始抓取 0092